In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

### Transformacion con pyspark

In [0]:
#movies_df = spark.read.parquet(f"{silver_folder_path}/movies")
movies_df = spark.read.table("movie_silver.movies")\
                        .filter(f"file_date = '{v_file_date}'")

#genre_df = spark.read.parquet(f"{silver_folder_path}/genre")
genre_df = spark.read.table("movie_silver.genres")

#movie_genre_df = spark.read.parquet(f"{silver_folder_path}/movie_genres")
movie_genre_df = spark.read.table("movie_silver.movie_genres")\
                        .filter(f"file_date = '{v_file_date}'")

#languages_df = spark.read.parquet(f"{silver_folder_path}/language")
languages_df = spark.read.table("movie_silver.languages")

#movies_languages_df = spark.read.parquet(f"{silver_folder_path}/movies_languages")
movies_languages_df = spark.read.table("movie_silver.movies_languages")\
                        .filter(f"file_date = '{v_file_date}'")

###Join "languages" y "movies_languages"

In [0]:
filter_movies_df = movies_df.filter(movies_df.year_release_date >= 2000)\
                    .select(movies_df.movie_id, 
                            movies_df.title,
                            movies_df.duration_time,
                            movies_df.release_date,
                            movies_df.vote_average)

In [0]:
movies_genre_genre_df = genre_df.join(movie_genre_df,
                                genre_df.genre_id == movie_genre_df.genre_id)\
                                .select(movie_genre_df.movie_id, genre_df.genre_name, 
                                        genre_df.genre_id)
                                

In [0]:
movies_languages_languages_df = languages_df.join(movies_languages_df,
                                                  languages_df.language_id == movies_languages_df.language_id)\
                                            .select(movies_languages_df.movie_id, languages_df.language_id, 
                                                    languages_df.language_name)

In [0]:
movies_inner_genre_df = filter_movies_df.join(movies_genre_genre_df,
                                          filter_movies_df.movie_id == movies_genre_genre_df.movie_id)\
                                        .drop(movies_genre_genre_df.movie_id)

In [0]:
from pyspark.sql.functions import lit
movie_genre_language_df = movies_inner_genre_df.join(movies_languages_languages_df,
                                                    movies_inner_genre_df.movie_id == movies_languages_languages_df.movie_id)\
                                                .orderBy(movies_inner_genre_df.release_date.desc())\
                                                .drop(#movies_inner_genre_df.movie_id,# 
                                                      movies_languages_languages_df.movie_id)\
                                                .withColumn("created_date", lit(v_file_date))

In [0]:
#overwrite_partition("movie_gold", "results_movie_genre_language","created_date", v_file_date)

In [0]:
# movie_genre_language_df.write.mode("overwrite").parquet(f"{gold_folder_path}/results_movie_genre_language")

#movie_genre_language_df.write.mode("append").partitionBy("created_date").format("delta").saveAsTable("movie_gold.results_movie_genre_language")

condition_merge = 'tgt.movie_id = src.movie_id AND tgt.genre_id = src.genre_id AND tgt.language_id = src.language_id AND tgt.created_date = src.created_date'

incremental_merge("movie_gold", "results_movie_genre_language", movie_genre_language_df, condition_merge, "created_date")

In [0]:
%sql
SELECT * FROM movie_gold.results_movie_genre_language

movie_id,title,duration_time,release_date,vote_average,genre_name,genre_id,language_id,language_name,created_date
426469,Growing Up Smith,102,2017-02-03,7.4,Drama,18,24574,English,2024-12-30
426469,Growing Up Smith,102,2017-02-03,7.4,Comedy,35,24574,English,2024-12-30
426469,Growing Up Smith,102,2017-02-03,7.4,Family,10751,24574,English,2024-12-30
325373,Two Lovers and a Bear,96,2016-10-02,6.8,Drama,18,24574,English,2024-12-30
325373,Two Lovers and a Bear,96,2016-10-02,6.8,Romance,10749,24574,English,2024-12-30
374461,Mr. Church,104,2016-09-16,7.0,Drama,18,24574,English,2024-12-30
339408,The Birth of a Nation,120,2016-09-09,6.5,Drama,18,24574,English,2024-12-30
385736,Kicks,80,2016-09-09,7.5,Adventure,12,24574,English,2024-12-30
332285,Antibirth,94,2016-09-02,4.8,Horror,27,24574,English,2024-12-30
184341,Hands of Stone,105,2016-08-26,6.1,Drama,18,24574,English,2024-12-30
